# Example 1.2.1 / 3: Default Values

Companion notebook for **1.2.1 Restrictions, Defaults and Checks** in the AMPL lab Documentation, section "Default Values". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Default Values

`default expr` gives the value to use for any instance the data doesn't mention. The data
then only has to list the exceptions, and from Python the dictionary lists only those. Like
`display rate;`, `to_pandas()` returns only the values the data gave. To get every value,
defaults included, ask for the expression over the indexing set:
`ampl.get_data("{p in PROD} rate[p]")` returns it as a `DataFrame`. `has_default()` tells
whether the declaration has a `default`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                            # products
param rate {PROD} > 0 default 100;   # tons per hour, unless the data says otherwise

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["rate"] = {"coils": 140}

In [5]:
# run.py
print(ampl.param["rate"].to_pandas())
print(ampl.get_data("{p in PROD} rate[p]").to_pandas())
print(ampl.param["rate"].has_default())

       rate
coils   140
       rate[p]
bands      100
coils      140
plate      100
True


### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param rate := coils 140;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
display rate;
display {p in PROD} rate[p];

rate [*] :=
coils  140
;

rate[p] [*] :=
bands  100
coils  140
plate  100
;

